In [39]:
# ============================================================
# CELL 1 — Imports and Configuration
# Quantum Portfolio Optimization using VQE
# ============================================================

# -----------------------------
# Standard library
# -----------------------------
from dataclasses import dataclass
from itertools import product
from typing import Optional, Callable, Dict, List, Tuple, Any


# -----------------------------
# Numerical / data libraries
# -----------------------------
import numpy as np
import pandas as pd

from scipy.optimize import minimize


# -----------------------------
# Visualization
# -----------------------------
import matplotlib.pyplot as plt


# -----------------------------
# Qiskit
# -----------------------------
import qiskit

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

from qiskit.quantum_info import (
    SparsePauliOp,
    Statevector
)

from qiskit.primitives import (
    StatevectorEstimator,
    StatevectorSampler
)


# ============================================================
# Reproducibility
# ============================================================

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)


# ============================================================
# Global numerical settings
# ============================================================

NUMERICAL_TOLERANCE = 1e-8

np.set_printoptions(
    precision=6,
    suppress=True
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.8f}"
)


# ============================================================
# Toy Portfolio Configuration
# ============================================================

NUM_ASSETS = 4
NUM_SELECTED = 2

RISK_AVERSION = 0.5
CONSTRAINT_PENALTY = 1.0


# ============================================================
# VQE Configuration
# ============================================================

ANSATZ_LAYERS = 2

OPTIMIZER = "L-BFGS-B"

MAX_ITERATIONS = 1000
VQE_TOLERANCE = 1e-6

INITIAL_PARAMETERS = "random"

SHOTS = 4096


# ============================================================
# Display configuration
# ============================================================

print("Environment configured successfully.")
print(f"Qiskit version        : {qiskit.__version__}")
print(f"Random seed            : {RANDOM_SEED}")
print(f"Number of assets       : {NUM_ASSETS}")
print(f"Assets to select (K)  : {NUM_SELECTED}")
print(f"Risk-aversion lambda  : {RISK_AVERSION}")
print(f"Constraint penalty P  : {CONSTRAINT_PENALTY}")
print(f"Ansatz layers          : {ANSATZ_LAYERS}")
print(f"Optimizer              : {OPTIMIZER}")
print(f"Maximum iterations     : {MAX_ITERATIONS}")
print(f"VQE tolerance           : {VQE_TOLERANCE}")
print(f"Sampling shots         : {SHOTS}")

Environment configured successfully.
Qiskit version        : 2.5.0
Random seed            : 42
Number of assets       : 4
Assets to select (K)  : 2
Risk-aversion lambda  : 0.5
Constraint penalty P  : 1.0
Ansatz layers          : 2
Optimizer              : L-BFGS-B
Maximum iterations     : 1000
VQE tolerance           : 1e-06
Sampling shots         : 4096


In [3]:
# ============================================================
# CELL 2 — Portfolio Problem Definition
# ============================================================

@dataclass
class PortfolioProblem:
    """
    Reusable representation of a binary portfolio-selection problem.

    Mathematical objective:

        C(x) = lambda_ * x^T Sigma x
               - mu^T x
               + penalty * (sum(x) - K)^2

    where:
        x_i = 1  -> asset i is selected
        x_i = 0  -> asset i is not selected
    """

    mu: np.ndarray
    Sigma: np.ndarray
    K: int
    lambda_: float
    penalty: float
    asset_names: Optional[List[str]] = None

    def __post_init__(self):
        # Convert inputs to NumPy arrays
        self.mu = np.asarray(self.mu, dtype=float)
        self.Sigma = np.asarray(self.Sigma, dtype=float)

        # -----------------------------
        # Validate expected returns
        # -----------------------------
        if self.mu.ndim != 1:
            raise ValueError(
                "`mu` must be a one-dimensional array."
            )

        # Number of assets
        self.num_assets = len(self.mu)

        # -----------------------------
        # Validate covariance matrix
        # -----------------------------
        if self.Sigma.ndim != 2:
            raise ValueError(
                "`Sigma` must be a two-dimensional matrix."
            )

        if self.Sigma.shape != (
            self.num_assets,
            self.num_assets
        ):
            raise ValueError(
                "`Sigma` must have shape "
                f"({self.num_assets}, {self.num_assets}). "
                f"Received {self.Sigma.shape}."
            )

        # Covariance matrix should be symmetric
        if not np.allclose(
            self.Sigma,
            self.Sigma.T,
            atol=NUMERICAL_TOLERANCE
        ):
            raise ValueError(
                "`Sigma` must be symmetric."
            )

        # -----------------------------
        # Validate K
        # -----------------------------
        if not isinstance(self.K, (int, np.integer)):
            raise TypeError("`K` must be an integer.")

        if not 0 <= self.K <= self.num_assets:
            raise ValueError(
                "`K` must satisfy 0 <= K <= number of assets."
            )

        # -----------------------------
        # Validate objective parameters
        # -----------------------------
        if self.lambda_ < 0:
            raise ValueError(
                "`lambda_` must be non-negative."
            )

        if self.penalty < 0:
            raise ValueError(
                "`penalty` must be non-negative."
            )

        # -----------------------------
        # Asset names
        # -----------------------------
        if self.asset_names is None:
            self.asset_names = [
                f"Asset_{i}"
                for i in range(self.num_assets)
            ]
        else:
            if len(self.asset_names) != self.num_assets:
                raise ValueError(
                    "`asset_names` must have one name per asset."
                )

            self.asset_names = list(self.asset_names)

    def summary(self):
        """
        Display a compact summary of the portfolio problem.
        """

        print("=" * 60)
        print("Portfolio Problem")
        print("=" * 60)

        print(f"Number of assets     : {self.num_assets}")
        print(f"Assets to select (K) : {self.K}")
        print(f"Risk aversion        : {self.lambda_}")
        print(f"Penalty              : {self.penalty}")

        print("\nExpected Returns:")
        print(
            pd.Series(
                self.mu,
                index=self.asset_names,
                name="mu"
            )
        )

        print("\nCovariance Matrix:")
        print(
            pd.DataFrame(
                self.Sigma,
                index=self.asset_names,
                columns=self.asset_names
            )
        )

        print("=" * 60)

In [4]:
# ============================================================
# 4-Asset Toy Portfolio
# ============================================================

mu = np.array([
    0.10,
    0.20,
    0.15,
    0.12
])

Sigma = np.array([
    [0.0400, 0.0060, 0.0100, 0.0080],
    [0.0060, 0.0900, 0.0120, 0.0070],
    [0.0100, 0.0120, 0.0625, 0.0090],
    [0.0080, 0.0070, 0.0090, 0.0500]
])

portfolio = PortfolioProblem(
    mu=mu,
    Sigma=Sigma,
    K=NUM_SELECTED,
    lambda_=RISK_AVERSION,
    penalty=CONSTRAINT_PENALTY,
    asset_names=["A0", "A1", "A2", "A3"]
)

portfolio.summary()

Portfolio Problem
Number of assets     : 4
Assets to select (K) : 2
Risk aversion        : 0.5
Penalty              : 1.0

Expected Returns:
A0   0.10000000
A1   0.20000000
A2   0.15000000
A3   0.12000000
Name: mu, dtype: float64

Covariance Matrix:
           A0         A1         A2         A3
A0 0.04000000 0.00600000 0.01000000 0.00800000
A1 0.00600000 0.09000000 0.01200000 0.00700000
A2 0.01000000 0.01200000 0.06250000 0.00900000
A3 0.00800000 0.00700000 0.00900000 0.05000000


In [5]:
# ============================================================
# CELL 3 — Classical Portfolio Cost Function
# ============================================================

def calculate_portfolio_cost(
    x: np.ndarray,
    problem: PortfolioProblem
) -> Dict[str, Any]:
    """
    Calculate all components of the classical portfolio objective.

    Objective:
        C(x) = lambda_ * x^T Sigma x
               - mu^T x
               + penalty * (sum(x) - K)^2

    Parameters
    ----------
    x : np.ndarray
        Binary portfolio-selection vector.

    problem : PortfolioProblem
        Portfolio problem definition.

    Returns
    -------
    Dict[str, Any]
        Dictionary containing:
            expected_return
            risk
            selected_count
            constraint_value
            penalty_cost
            total_cost
    """

    x = np.asarray(x, dtype=int)

    # --------------------------------------------------------
    # Validate portfolio vector
    # --------------------------------------------------------

    if x.shape != (problem.num_assets,):
        raise ValueError(
            f"`x` must have shape ({problem.num_assets},). "
            f"Received {x.shape}."
        )

    if not np.all(np.isin(x, [0, 1])):
        raise ValueError(
            "`x` must contain only binary values 0 or 1."
        )

    # --------------------------------------------------------
    # Expected return
    #       mu^T x
    # --------------------------------------------------------

    expected_return = float(
        problem.mu @ x
    )

    # --------------------------------------------------------
    # Portfolio risk
    #       x^T Sigma x
    # --------------------------------------------------------

    risk = float(
        x @ problem.Sigma @ x
    )

    # --------------------------------------------------------
    # Cardinality constraint
    #       sum(x) = K
    # --------------------------------------------------------

    selected_count = int(
        np.sum(x)
    )

    constraint_value = (
        selected_count - problem.K
    )

    # --------------------------------------------------------
    # Constraint penalty
    #       P (sum(x) - K)^2
    # --------------------------------------------------------

    penalty_cost = float(
        problem.penalty * constraint_value**2
    )

    # --------------------------------------------------------
    # Complete objective
    #       lambda * risk - return + penalty
    # --------------------------------------------------------

    total_cost = float(
        problem.lambda_ * risk
        - expected_return
        + penalty_cost
    )

    return {
        "expected_return": expected_return,
        "risk": risk,
        "selected_count": selected_count,
        "constraint_value": constraint_value,
        "penalty_cost": penalty_cost,
        "total_cost": total_cost,
    }

In [6]:
# ============================================================
# Test: Portfolio A0 + A2
# ============================================================

x_test = np.array([1, 0, 1, 0])

cost_test = calculate_portfolio_cost(
    x=x_test,
    problem=portfolio
)

print("Portfolio:", x_test)
print()

for key, value in cost_test.items():
    print(f"{key:20s}: {value}")

# Pretty summary for the test portfolio

print(f"Selected assets     : "
      f"{[portfolio.asset_names[i] for i, bit in enumerate(x_test) if bit == 1]}")

print(f"Expected return     : {cost_test['expected_return']:.6f}")
print(f"Risk                : {cost_test['risk']:.6f}")
print(f"Selected count      : {cost_test['selected_count']}")
print(f"Constraint value    : {cost_test['constraint_value']}")
print(f"Penalty             : {cost_test['penalty_cost']:.6f}")
print(f"Total objective     : {cost_test['total_cost']:.6f}")

Portfolio: [1 0 1 0]

expected_return     : 0.25
risk                : 0.1225
selected_count      : 2
constraint_value    : 0
penalty_cost        : 0.0
total_cost          : -0.18875
Selected assets     : ['A0', 'A2']
Expected return     : 0.250000
Risk                : 0.122500
Selected count      : 2
Constraint value    : 0
Penalty             : 0.000000
Total objective     : -0.188750


In [7]:
# ============================================================
# CELL 4 — Exact Classical Solver
# ============================================================

def solve_classically(
    problem: PortfolioProblem,
    sort_results: bool = True
) -> pd.DataFrame:
    """
    Exhaustively evaluate all binary portfolios.

    For N assets, all 2^N possible portfolios are evaluated.

    Parameters
    ----------
    problem : PortfolioProblem
        Portfolio problem definition.

    sort_results : bool, default=True
        Sort portfolios by increasing objective value.

    Returns
    -------
    pd.DataFrame
        Complete evaluation of all binary portfolios.
    """

    results = []

    # --------------------------------------------------------
    # Enumerate all 2^N binary portfolios
    # --------------------------------------------------------

    for bits in product([0, 1], repeat=problem.num_assets):

        x = np.array(bits, dtype=int)

        cost_data = calculate_portfolio_cost(
            x=x,
            problem=problem
        )

        selected_assets = [
            problem.asset_names[i]
            for i, bit in enumerate(x)
            if bit == 1
        ]

        results.append({
            "bitstring": "".join(map(str, x)),
            "selected_assets": ", ".join(selected_assets)
                              if selected_assets else "None",
            "expected_return": cost_data["expected_return"],
            "risk": cost_data["risk"],
            "selected_count": cost_data["selected_count"],
            "constraint_value": cost_data["constraint_value"],
            "penalty": cost_data["penalty_cost"],
            "objective": cost_data["total_cost"],
        })

    results_df = pd.DataFrame(results)

    # --------------------------------------------------------
    # Sort by objective value
    # --------------------------------------------------------

    if sort_results:
        results_df = (
            results_df
            .sort_values(
                by="objective",
                ascending=True
            )
            .reset_index(drop=True)
        )

    return results_df


def get_classical_optimum(
    classical_results: pd.DataFrame
) -> Dict[str, Any]:
    """
    Extract the exact classical optimum from exhaustive results.
    """

    optimum = classical_results.iloc[0]

    return {
        "bitstring": optimum["bitstring"],
        "selected_assets": optimum["selected_assets"],
        "expected_return": float(optimum["expected_return"]),
        "risk": float(optimum["risk"]),
        "selected_count": int(optimum["selected_count"]),
        "constraint_value": float(optimum["constraint_value"]),
        "penalty": float(optimum["penalty"]),
        "objective": float(optimum["objective"]),
    }

In [8]:
# ============================================================
# Run Exact Classical Solver
# ============================================================

classical_results = solve_classically(
    problem=portfolio
)

print(f"Number of portfolios evaluated: {len(classical_results)}")

display(classical_results)

# ============================================================
# Exact Classical Ground Truth
# ============================================================

classical_optimum = get_classical_optimum(
    classical_results
)

print("=" * 60)
print("EXACT CLASSICAL OPTIMUM")
print("=" * 60)

print(f"Bitstring          : {classical_optimum['bitstring']}")
print(f"Selected assets    : {classical_optimum['selected_assets']}")
print(f"Expected return    : {classical_optimum['expected_return']:.6f}")
print(f"Risk               : {classical_optimum['risk']:.6f}")
print(f"Selected count     : {classical_optimum['selected_count']}")
print(f"Constraint value   : {classical_optimum['constraint_value']:.6f}")
print(f"Penalty            : {classical_optimum['penalty']:.6f}")
print(f"Objective          : {classical_optimum['objective']:.6f}")

Number of portfolios evaluated: 16


,bitstring,selected_assets,expected_return,risk,selected_count,constraint_value,penalty,objective
0,0110,"A1, A2",0.35000000,0.17650000,2,0,0.00000000,-0.26175000
1,0101,"A1, A3",0.32000000,0.15400000,2,0,0.00000000,-0.24300000
2,1100,"A0, A1",0.30000000,0.14200000,2,0,0.00000000,-0.22900000
3,0011,"A2, A3",0.27000000,0.13050000,2,0,0.00000000,-0.20475000
4,1010,"A0, A2",0.25000000,0.12250000,2,0,0.00000000,-0.18875000
5,1001,"A0, A3",0.22000000,0.10600000,2,0,0.00000000,-0.16700000
6,0111,"A1, A2, A3",0.47000000,0.25850000,3,1,1.00000000,0.65925000
7,1110,"A0, A1, A2",0.45000000,0.24850000,3,1,1.00000000,0.67425000
8,1101,"A0, A1, A3",0.42000000,0.22200000,3,1,1.00000000,0.69100000
9,1011,"A0, A2, A3",0.37000000,0.20650000,3,1,1.00000000,0.73325000


EXACT CLASSICAL OPTIMUM
Bitstring          : 0110
Selected assets    : A1, A2
Expected return    : 0.350000
Risk               : 0.176500
Selected count     : 2
Constraint value   : 0.000000
Penalty            : 0.000000
Objective          : -0.261750


In [11]:
# ============================================================
# Feasible Portfolios Only
# ============================================================

feasible_results = classical_results[
    classical_results["selected_count"] == portfolio.K
].copy()

print(
    f"Number of feasible portfolios: "
    f"{len(feasible_results)}"
)

display(feasible_results)

# ============================================================
# Classical Solver Sanity Checks
# ============================================================

from math import comb

expected_number_of_portfolios = 2 ** portfolio.num_assets

assert len(classical_results) == expected_number_of_portfolios

expected_number_of_feasible = comb(
    portfolio.num_assets,
    portfolio.K
)

assert len(feasible_results) == expected_number_of_feasible

print("Classical exhaustive-search validation passed.")
print(f"Total portfolios evaluated : {len(classical_results)}")
print(f"Feasible portfolios        : {len(feasible_results)}")

Number of feasible portfolios: 6


,bitstring,selected_assets,expected_return,risk,selected_count,constraint_value,penalty,objective
0,0110,"A1, A2",0.35000000,0.17650000,2,0,0.00000000,-0.26175000
1,0101,"A1, A3",0.32000000,0.15400000,2,0,0.00000000,-0.24300000
2,1100,"A0, A1",0.30000000,0.14200000,2,0,0.00000000,-0.22900000
3,0011,"A2, A3",0.27000000,0.13050000,2,0,0.00000000,-0.20475000
4,1010,"A0, A2",0.25000000,0.12250000,2,0,0.00000000,-0.18875000
5,1001,"A0, A3",0.22000000,0.10600000,2,0,0.00000000,-0.16700000


Classical exhaustive-search validation passed.
Total portfolios evaluated : 16
Feasible portfolios        : 6


In [13]:
# ============================================================
# CELL 5 — Direct Cost Hamiltonian Construction
# ============================================================

def _z_pauli_label(
    qubit_index: int,
    num_qubits: int
) -> str:
    """
    Create a Qiskit Pauli label containing Z on the
    requested qubit and I on all other qubits.

    Qiskit Pauli strings are ordered as:

        q_(n-1) ... q_2 q_1 q_0

    Therefore qubit i is placed at position n - 1 - i.
    """

    label = ["I"] * num_qubits
    label[num_qubits - 1 - qubit_index] = "Z"

    return "".join(label)


def _zz_pauli_label(
    qubit_i: int,
    qubit_j: int,
    num_qubits: int
) -> str:
    """
    Create a Qiskit Pauli label containing Z on qubits i and j.
    """

    label = ["I"] * num_qubits

    label[num_qubits - 1 - qubit_i] = "Z"
    label[num_qubits - 1 - qubit_j] = "Z"

    return "".join(label)


def build_cost_hamiltonian(
    problem: PortfolioProblem
) -> Tuple[SparsePauliOp, float, np.ndarray, np.ndarray]:
    """
    Construct the Ising cost Hamiltonian directly from the
    classical portfolio objective.

    Classical objective:

        C(x) = lambda * x^T Sigma x
               - mu^T x
               + P * (sum_i x_i - K)^2

    Binary-to-spin transformation:

        x_i = (1 - Z_i) / 2

    Resulting Hamiltonian:

        H_C = c * I
              + sum_i h_i Z_i
              + sum_{i<j} J_ij Z_i Z_j

    Returns
    -------
    hamiltonian : SparsePauliOp
        Qiskit cost Hamiltonian.

    constant : float
        Identity coefficient c.

    h_coefficients : np.ndarray
        Single-qubit Z coefficients h_i.

    interaction_coefficients : np.ndarray
        Symmetric matrix containing J_ij.
    """

    n = problem.num_assets

    lambda_ = problem.lambda_
    penalty = problem.penalty
    K = problem.K

    # --------------------------------------------------------
    # First collect the binary polynomial coefficients
    #
    # C(x) = constant
    #        + sum_i a_i x_i
    #        + sum_{i<j} b_ij x_i x_j
    # --------------------------------------------------------

    # Linear coefficients:
    #
    # lambda * Sigma_ii
    # - mu_i
    # + P(1 - 2K)

    linear_coefficients = (
        lambda_ * np.diag(problem.Sigma)
        - problem.mu
        + penalty * (1 - 2 * K)
    )

    # Pair coefficients:
    #
    # 2 * lambda * Sigma_ij
    # + 2 * P
    #
    # because x^T Sigma x contains both (i,j) and (j,i).

    pair_coefficients = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(i + 1, n):

            pair_coefficient = (
                2.0 * lambda_ * problem.Sigma[i, j]
                + 2.0 * penalty
            )

            pair_coefficients[i, j] = pair_coefficient
            pair_coefficients[j, i] = pair_coefficient

    # --------------------------------------------------------
    # Binary -> Ising transformation
    #
    # x_i = (1 - Z_i) / 2
    #
    # x_i x_j =
    #     (1 - Z_i - Z_j + Z_i Z_j) / 4
    # --------------------------------------------------------

    h_coefficients = np.zeros(n, dtype=float)
    interaction_coefficients = np.zeros((n, n), dtype=float)

    # Identity contribution
    constant = penalty * (K ** 2)

    # Linear terms transformed to Ising form
    for i in range(n):

        a_i = linear_coefficients[i]

        constant += a_i / 2.0
        h_coefficients[i] += -a_i / 2.0

    # Pair terms transformed to Ising form
    for i in range(n):
        for j in range(i + 1, n):

            b_ij = pair_coefficients[i, j]

            constant += b_ij / 4.0

            h_coefficients[i] += -b_ij / 4.0
            h_coefficients[j] += -b_ij / 4.0

            interaction_coefficients[i, j] = b_ij / 4.0
            interaction_coefficients[j, i] = b_ij / 4.0

    # --------------------------------------------------------
    # Build Qiskit SparsePauliOp
    # --------------------------------------------------------

    pauli_terms = []

    # Identity term
    pauli_terms.append(
        ("I" * n, constant)
    )

    # Single-qubit Z terms
    for i in range(n):

        coefficient = h_coefficients[i]

        if not np.isclose(
            coefficient,
            0.0,
            atol=NUMERICAL_TOLERANCE
        ):
            pauli_terms.append(
                (
                    _z_pauli_label(i, n),
                    coefficient
                )
            )

    # Two-qubit ZZ interaction terms
    for i in range(n):
        for j in range(i + 1, n):

            coefficient = interaction_coefficients[i, j]

            if not np.isclose(
                coefficient,
                0.0,
                atol=NUMERICAL_TOLERANCE
            ):
                pauli_terms.append(
                    (
                        _zz_pauli_label(i, j, n),
                        coefficient
                    )
                )

    hamiltonian = SparsePauliOp.from_list(
        pauli_terms
    )

    # Combine any duplicate Pauli terms
    hamiltonian = hamiltonian.simplify()

    return (
        hamiltonian,
        float(constant),
        h_coefficients,
        interaction_coefficients
    )

In [14]:
# ============================================================
# Build Cost Hamiltonian
# ============================================================

(
    cost_hamiltonian,
    hamiltonian_constant,
    h_coefficients,
    J_coefficients
) = build_cost_hamiltonian(
    problem=portfolio
)

print("Cost Hamiltonian:")
print(cost_hamiltonian)

print("\nIdentity coefficient:")
print(f"{hamiltonian_constant:.8f}")

print("\nSingle-qubit Z coefficients:")
for i, coefficient in enumerate(h_coefficients):
    print(
        f"q{i} ({portfolio.asset_names[i]}): "
        f"{coefficient:.8f}"
    )

print("\nZZ interaction coefficients:")
display(
    pd.DataFrame(
        J_coefficients,
        index=portfolio.asset_names,
        columns=portfolio.asset_names
    )
)

Cost Hamiltonian:
SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'IZIZ', 'ZIIZ', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[0.788625+0.j, 0.034   +0.j, 0.07125 +0.j, 0.051625+0.j, 0.0415  +0.j,
 0.5015  +0.j, 0.5025  +0.j, 0.502   +0.j, 0.503   +0.j, 0.50175 +0.j,
 0.50225 +0.j])

Identity coefficient:
0.78862500

Single-qubit Z coefficients:
q0 (A0): 0.03400000
q1 (A1): 0.07125000
q2 (A2): 0.05162500
q3 (A3): 0.04150000

ZZ interaction coefficients:


,A0,A1,A2,A3
A0,0.00000000,0.50150000,0.50250000,0.50200000
A1,0.50150000,0.00000000,0.50300000,0.50175000
A2,0.50250000,0.50300000,0.00000000,0.50225000
A3,0.50200000,0.50175000,0.50225000,0.00000000


In [15]:
# ============================================================
# CELL 6 — Hamiltonian Validation
# ============================================================

def validate_hamiltonian(
    problem: PortfolioProblem,
    hamiltonian: SparsePauliOp,
    tolerance: float = NUMERICAL_TOLERANCE
) -> pd.DataFrame:
    """
    Validate that the cost Hamiltonian reproduces the
    classical portfolio objective for every binary state.

    Required relationship:

        <x|H_C|x> = C(x)

    Parameters
    ----------
    problem : PortfolioProblem
        Portfolio problem definition.

    hamiltonian : SparsePauliOp
        Ising cost Hamiltonian.

    tolerance : float
        Numerical tolerance for validation.

    Returns
    -------
    pd.DataFrame
        Validation table for all computational-basis states.

    Raises
    ------
    ValueError
        If any Hamiltonian energy differs from the corresponding
        classical cost by more than the specified tolerance.
    """

    n = problem.num_assets

    # Convert the SparsePauliOp into a dense matrix.
    #
    # This is perfectly reasonable for the current 4-qubit
    # toy problem and makes the validation transparent.
    H_matrix = hamiltonian.to_matrix()

    validation_rows = []

    # --------------------------------------------------------
    # Check every computational-basis state
    # --------------------------------------------------------

    for bits in product([0, 1], repeat=n):

        x = np.array(bits, dtype=int)

        # Classical objective
        classical_data = calculate_portfolio_cost(
            x=x,
            problem=problem
        )

        classical_cost = classical_data["total_cost"]

        # ----------------------------------------------------
        # Qiskit's computational basis convention
        #
        # Statevector basis index uses:
        #
        # |q_(n-1) ... q_1 q_0>
        #
        # while our x vector is stored as:
        #
        # [x_0, x_1, ..., x_(n-1)]
        #
        # Therefore reverse the bitstring when converting
        # to a computational-basis matrix index.
        # ----------------------------------------------------

        bitstring = "".join(map(str, x))

        qiskit_bitstring = bitstring[::-1]

        basis_index = int(
            qiskit_bitstring,
            2
        )

        basis_state = np.zeros(
            2 ** n,
            dtype=complex
        )

        basis_state[basis_index] = 1.0

        # Hamiltonian energy:
        #
        # <x|H|x>
        hamiltonian_energy = np.vdot(
            basis_state,
            H_matrix @ basis_state
        ).real

        difference = (
            hamiltonian_energy
            - classical_cost
        )

        validation_rows.append({
            "bitstring": bitstring,
            "classical_cost": classical_cost,
            "hamiltonian_energy": hamiltonian_energy,
            "difference": difference,
            "valid": np.isclose(
                hamiltonian_energy,
                classical_cost,
                atol=tolerance
            )
        })

    validation_df = pd.DataFrame(
        validation_rows
    )

    # --------------------------------------------------------
    # Overall validation
    # --------------------------------------------------------

    max_error = np.max(
        np.abs(validation_df["difference"])
    )

    all_valid = bool(
        np.all(validation_df["valid"])
    )

    print("=" * 65)
    print("HAMILTONIAN VALIDATION")
    print("=" * 65)

    print(
        f"States tested : {len(validation_df)}"
    )

    print(
        f"Maximum error : {max_error:.12e}"
    )

    print(
        f"Tolerance     : {tolerance:.12e}"
    )

    print(
        f"Validation    : {'PASSED' if all_valid else 'FAILED'}"
    )

    print("=" * 65)

    if not all_valid:
        raise ValueError(
            "Hamiltonian validation failed. "
            "The Hamiltonian does not reproduce the "
            "classical portfolio objective within tolerance."
        )

    return validation_df

In [16]:
# ============================================================
# Run Hamiltonian Validation
# ============================================================

hamiltonian_validation = validate_hamiltonian(
    problem=portfolio,
    hamiltonian=cost_hamiltonian
)

display(
    hamiltonian_validation
)

HAMILTONIAN VALIDATION
States tested : 16
Maximum error : 4.440892098501e-16
Tolerance     : 1.000000000000e-08
Validation    : PASSED


,bitstring,classical_cost,hamiltonian_energy,difference,valid
0,0000,4.00000000,4.00000000,0.00000000,True
1,0001,0.90500000,0.90500000,-0.00000000,True
2,0010,0.88125000,0.88125000,0.00000000,True
3,0011,-0.20475000,-0.20475000,-0.00000000,True
4,0100,0.84500000,0.84500000,-0.00000000,True
5,0101,-0.24300000,-0.24300000,-0.00000000,True
6,0110,-0.26175000,-0.26175000,-0.00000000,True
7,0111,0.65925000,0.65925000,-0.00000000,True
8,1000,0.92000000,0.92000000,0.00000000,True
9,1001,-0.16700000,-0.16700000,-0.00000000,True


In [17]:
# ============================================================
# Validation Summary
# ============================================================

assert len(hamiltonian_validation) == 2 ** portfolio.num_assets

assert np.all(
    hamiltonian_validation["valid"]
)

print(
    "\nAll computational-basis states reproduce the "
    "classical portfolio objective."
)


All computational-basis states reproduce the classical portfolio objective.


In [18]:
# ============================================================
# CELL 7 — Configurable VQE Ansatz
# ============================================================

def build_vqe_ansatz(
    num_qubits: int,
    num_layers: int = 2,
    rotation_gates: Tuple[str, ...] = ("ry",),
    entanglement: str = "linear"
) -> Tuple[QuantumCircuit, ParameterVector]:
    """
    Build a configurable hardware-efficient VQE ansatz.

    Structure for each layer:

        Single-qubit rotations
                 ↓
        Entangling gates

    Parameters
    ----------
    num_qubits : int
        Number of qubits.

    num_layers : int
        Number of variational layers.

    rotation_gates : tuple of str
        Rotation gates to apply to each qubit.
        Supported:
            "rx"
            "ry"
            "rz"

        For example:
            ("ry",)
            ("ry", "rz")

    entanglement : str
        Entanglement pattern.

        Supported:
            "linear"
            "ring"
            "full"
            "none"

    Returns
    -------
    circuit : QuantumCircuit
        Parameterized ansatz circuit.

    parameters : ParameterVector
        Variational parameters used by the circuit.
    """

    if num_qubits < 1:
        raise ValueError(
            "`num_qubits` must be at least 1."
        )

    if num_layers < 1:
        raise ValueError(
            "`num_layers` must be at least 1."
        )

    supported_rotations = {"rx", "ry", "rz"}

    invalid_rotations = (
        set(rotation_gates) - supported_rotations
    )

    if invalid_rotations:
        raise ValueError(
            f"Unsupported rotation gates: {invalid_rotations}. "
            f"Supported gates: {supported_rotations}"
        )

    supported_entanglement = {
        "linear",
        "ring",
        "full",
        "none"
    }

    if entanglement not in supported_entanglement:
        raise ValueError(
            f"Unsupported entanglement pattern: "
            f"{entanglement}. "
            f"Supported patterns: {supported_entanglement}"
        )

    # --------------------------------------------------------
    # Number of parameters
    # --------------------------------------------------------

    parameters_per_layer = (
        num_qubits * len(rotation_gates)
    )

    total_parameters = (
        num_layers * parameters_per_layer
    )

    parameters = ParameterVector(
        "theta",
        length=total_parameters
    )

    circuit = QuantumCircuit(
        num_qubits,
        name="VQE_Ansatz"
    )

    parameter_index = 0

    # --------------------------------------------------------
    # Build variational layers
    # --------------------------------------------------------

    for layer in range(num_layers):

        # --------------------------------------------
        # Single-qubit rotation layer
        # --------------------------------------------

        for gate_name in rotation_gates:

            for qubit in range(num_qubits):

                theta = parameters[
                    parameter_index
                ]

                if gate_name == "rx":
                    circuit.rx(theta, qubit)

                elif gate_name == "ry":
                    circuit.ry(theta, qubit)

                elif gate_name == "rz":
                    circuit.rz(theta, qubit)

                parameter_index += 1

        # --------------------------------------------
        # Entangling layer
        # --------------------------------------------

        if entanglement == "linear":

            for qubit in range(num_qubits - 1):
                circuit.cx(
                    qubit,
                    qubit + 1
                )

        elif entanglement == "ring":

            for qubit in range(num_qubits - 1):
                circuit.cx(
                    qubit,
                    qubit + 1
                )

            if num_qubits > 2:
                circuit.cx(
                    num_qubits - 1,
                    0
                )

        elif entanglement == "full":

            for control in range(num_qubits):
                for target in range(control + 1, num_qubits):
                    circuit.cx(
                        control,
                        target
                    )

        # No gates for "none"

    return circuit, parameters

In [19]:
# ============================================================
# Build the Toy-Problem VQE Ansatz
# ============================================================

ansatz, ansatz_parameters = build_vqe_ansatz(
    num_qubits=portfolio.num_assets,
    num_layers=ANSATZ_LAYERS,
    rotation_gates=("ry",),
    entanglement="linear"
)

print("Number of qubits:", ansatz.num_qubits)
print("Number of parameters:", len(ansatz_parameters))
print()
print(ansatz)

Number of qubits: 4
Number of parameters: 8

     ┌──────────────┐     ┌──────────────┐                                     »
q_0: ┤ Ry(theta[0]) ├──■──┤ Ry(theta[4]) ├───────────────────────■─────────────»
     ├──────────────┤┌─┴─┐└──────────────┘┌──────────────┐     ┌─┴─┐           »
q_1: ┤ Ry(theta[1]) ├┤ X ├───────■────────┤ Ry(theta[5]) ├─────┤ X ├────────■──»
     ├──────────────┤└───┘     ┌─┴─┐      └──────────────┘┌────┴───┴─────┐┌─┴─┐»
q_2: ┤ Ry(theta[2]) ├──────────┤ X ├─────────────■────────┤ Ry(theta[6]) ├┤ X ├»
     ├──────────────┤          └───┘           ┌─┴─┐      ├──────────────┤└───┘»
q_3: ┤ Ry(theta[3]) ├──────────────────────────┤ X ├──────┤ Ry(theta[7]) ├─────»
     └──────────────┘                          └───┘      └──────────────┘     »
«          
«q_0: ─────
«          
«q_1: ─────
«          
«q_2: ──■──
«     ┌─┴─┐
«q_3: ┤ X ├
«     └───┘


In [20]:
# ============================================================
# Ansatz Parameters
# ============================================================

print("Variational parameters:")
print(ansatz_parameters)

print("\nParameter count:")
print(len(ansatz_parameters))

Variational parameters:
theta, ['theta[0]', 'theta[1]', 'theta[2]', 'theta[3]', 'theta[4]', 'theta[5]', 'theta[6]', 'theta[7]']

Parameter count:
8


In [21]:
# ============================================================
# CELL 8 — VQE Initial Parameters
# ============================================================

def initialize_vqe_parameters(
    num_parameters: int,
    method: str = "random",
    seed: Optional[int] = None,
    initial_parameters: Optional[np.ndarray] = None
) -> np.ndarray:
    """
    Generate the initial variational parameters for VQE.

    Parameters
    ----------
    num_parameters : int
        Number of variational parameters in the ansatz.

    method : str
        Initialization method.

        Supported:
            "random"
            "zeros"

    seed : int, optional
        Random seed used for reproducibility.

    initial_parameters : np.ndarray, optional
        Explicit user-provided initial parameter vector.
        When supplied, this takes precedence over `method`.

    Returns
    -------
    np.ndarray
        Initial parameter vector.
    """

    # --------------------------------------------------------
    # User-provided parameters
    # --------------------------------------------------------

    if initial_parameters is not None:

        parameters = np.asarray(
            initial_parameters,
            dtype=float
        )

        if parameters.ndim != 1:
            raise ValueError(
                "`initial_parameters` must be a "
                "one-dimensional array."
            )

        if len(parameters) != num_parameters:
            raise ValueError(
                f"Expected {num_parameters} initial parameters, "
                f"but received {len(parameters)}."
            )

        return parameters.copy()

    # --------------------------------------------------------
    # Validate initialization method
    # --------------------------------------------------------

    supported_methods = {
        "random",
        "zeros"
    }

    if method not in supported_methods:
        raise ValueError(
            f"Unsupported initialization method: {method}. "
            f"Supported methods: {supported_methods}"
        )

    # --------------------------------------------------------
    # Initialize parameters
    # --------------------------------------------------------

    if method == "zeros":

        parameters = np.zeros(
            num_parameters,
            dtype=float
        )

    elif method == "random":

        rng = np.random.default_rng(seed)

        parameters = rng.uniform(
            low=-np.pi,
            high=np.pi,
            size=num_parameters
        )

    return parameters

In [22]:
# ============================================================
# Initialize Parameters for Current Ansatz
# ============================================================

initial_parameters = initialize_vqe_parameters(
    num_parameters=len(ansatz_parameters),
    method=INITIAL_PARAMETERS,
    seed=RANDOM_SEED
)

print("Initial VQE parameters:")
print(initial_parameters)

print("\nNumber of parameters:")
print(len(initial_parameters))

# ============================================================
# Reproducibility Check
# ============================================================

initial_parameters_check = initialize_vqe_parameters(
    num_parameters=len(ansatz_parameters),
    method=INITIAL_PARAMETERS,
    seed=RANDOM_SEED
)

assert np.allclose(
    initial_parameters,
    initial_parameters_check
)

print("Initial parameter generation is reproducible.")


custom_parameters = np.zeros(
    len(ansatz_parameters)
)

initial_parameters = initialize_vqe_parameters(
    num_parameters=len(ansatz_parameters),
    initial_parameters=custom_parameters
)

Initial VQE parameters:
[ 1.721317 -0.384038  2.253137  1.2401   -2.549859  2.988423  1.640789
  1.797395]

Number of parameters:
8
Initial parameter generation is reproducible.


In [42]:
# ============================================================
# CELL 9 — VQE Energy Evaluation and Optimization
# ============================================================

def create_vqe_energy_function(
    ansatz: QuantumCircuit,
    parameters: ParameterVector,
    hamiltonian: SparsePauliOp,
    estimator: StatevectorEstimator
) -> Callable[[np.ndarray], float]:
    """
    Create the VQE energy function:

        E(theta) = <psi(theta)|H|psi(theta)>

    The function is compatible with Qiskit's current
    Estimator V2 / StatevectorEstimator interface.
    """

    def energy_function(theta: np.ndarray) -> float:

        theta = np.asarray(theta, dtype=float)

        # Validate parameter count
        if theta.ndim != 1:
            raise ValueError(
                "`theta` must be a one-dimensional array."
            )

        if len(theta) != len(parameters):
            raise ValueError(
                f"Expected {len(parameters)} parameters, "
                f"received {len(theta)}."
            )

        # ----------------------------------------------------
        # Create one PUB:
        #
        # (circuit, observable, parameter values)
        # ----------------------------------------------------

        job = estimator.run(
            [
                (
                    ansatz,
                    hamiltonian,
                    theta
                )
            ]
        )

        pub_result = job.result()[0]

        # ----------------------------------------------------
        # Current Qiskit may return:
        #
        #   scalar / 0-D array
        #
        # for a single expectation value.
        #
        # np.asarray(...).squeeze().item()
        # safely converts that to a Python scalar.
        # ----------------------------------------------------

        energy = np.asarray(
            pub_result.data.evs
        ).squeeze().item()

        return float(
            np.real(energy)
        )

    return energy_function


def run_vqe(
    ansatz: QuantumCircuit,
    parameters: ParameterVector,
    hamiltonian: SparsePauliOp,
    initial_parameters: np.ndarray,
    optimizer: str = "COBYLA",
    max_iterations: int = 100,
    tolerance: float = 1e-6,
    seed: Optional[int] = None
) -> Dict[str, Any]:
    """
    Run VQE using a configurable classical optimizer.

    VQE minimizes:

        E(theta) = <psi(theta)|H|psi(theta)>

    Returns a dictionary containing the optimized parameters,
    energy, convergence history, and optimizer information.
    """

    initial_parameters = np.asarray(
        initial_parameters,
        dtype=float
    )

    # --------------------------------------------------------
    # Validate initial parameters
    # --------------------------------------------------------

    if initial_parameters.ndim != 1:
        raise ValueError(
            "`initial_parameters` must be one-dimensional."
        )

    if len(initial_parameters) != len(parameters):
        raise ValueError(
            f"Expected {len(parameters)} initial parameters, "
            f"received {len(initial_parameters)}."
        )

    # --------------------------------------------------------
    # Statevector estimator
    # --------------------------------------------------------

    estimator = StatevectorEstimator(
        seed=seed
    )

    # --------------------------------------------------------
    # VQE energy function
    # --------------------------------------------------------

    energy_function = create_vqe_energy_function(
        ansatz=ansatz,
        parameters=parameters,
        hamiltonian=hamiltonian,
        estimator=estimator
    )

    # --------------------------------------------------------
    # Optimization history
    # --------------------------------------------------------

    energy_history = []
    parameter_history = []

    # --------------------------------------------------------
    # Objective passed to SciPy
    # --------------------------------------------------------

    def objective(theta):

        energy = energy_function(theta)

        energy_history.append(
            float(energy)
        )

        parameter_history.append(
            np.asarray(theta, dtype=float).copy()
        )

        return energy

    # --------------------------------------------------------
    # Select optimizer
    # --------------------------------------------------------

    optimizer_upper = optimizer.upper()

    if optimizer_upper == "COBYLA":

        optimizer_method = "COBYLA"

        optimizer_options = {
            "maxiter": max_iterations,
            "tol": tolerance
        }

    elif optimizer_upper == "POWELL":

        optimizer_method = "Powell"

        optimizer_options = {
            "maxiter": max_iterations,
            "xtol": tolerance,
            "ftol": tolerance
        }

    elif optimizer_upper == "NELDER-MEAD":

        optimizer_method = "Nelder-Mead"

        optimizer_options = {
            "maxiter": max_iterations,
            "xatol": tolerance,
            "fatol": tolerance
        }

    elif optimizer_upper == "L-BFGS-B":

        optimizer_method = "L-BFGS-B"
    
        optimizer_options = {
            "maxiter": max_iterations,
            "ftol": tolerance,
            "gtol": tolerance,
            "maxls": 50
        }

    else:

        raise ValueError(
            f"Unsupported optimizer: {optimizer}. "
            "Supported optimizers are: "
            "COBYLA, POWELL, NELDER-MEAD, BFGS."
        )

    # --------------------------------------------------------
    # Run classical optimization
    # --------------------------------------------------------

    optimization_result = minimize(
        fun=objective,
        x0=initial_parameters,
        method=optimizer_method,
        options=optimizer_options
    )

    # --------------------------------------------------------
    # Final optimized parameters
    # --------------------------------------------------------

    optimal_parameters = np.asarray(
        optimization_result.x,
        dtype=float
    )

    # Evaluate the final state once more
    minimum_energy = energy_function(
        optimal_parameters
    )

    # --------------------------------------------------------
    # Return structured results
    # --------------------------------------------------------

    return {
        "optimal_parameters": optimal_parameters,
        "minimum_energy": float(minimum_energy),

        "energy_history": energy_history,
        "parameter_history": parameter_history,

        "iterations": len(energy_history),

        "success": bool(
            optimization_result.success
        ),

        "message": str(
            optimization_result.message
        ),

        "optimizer_result": optimization_result,

        "estimator": estimator
    }

In [43]:
# ============================================================
# Run VQE
# ============================================================

vqe_results = run_vqe(
    ansatz=ansatz,
    parameters=ansatz_parameters,
    hamiltonian=cost_hamiltonian,
    initial_parameters=initial_parameters,
    optimizer=OPTIMIZER,
    max_iterations=MAX_ITERATIONS,
    tolerance=VQE_TOLERANCE,
    seed=RANDOM_SEED
)

print("=" * 60)
print("VQE OPTIMIZATION")
print("=" * 60)

print(f"Optimizer           : {OPTIMIZER}")
print(f"Iterations           : {vqe_results['iterations']}")
print(
    f"Minimum energy       : "
    f"{vqe_results['minimum_energy']:.10f}"
)
print(
    f"Optimization success : "
    f"{vqe_results['success']}"
)
print(
    f"Message              : "
    f"{vqe_results['message']}"
)

VQE OPTIMIZATION
Optimizer           : L-BFGS-B
Iterations           : 9
Minimum energy       : 4.0000000000
Optimization success : True
Message              : CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
